# Symbolic Einstein–Euler / BDNK calculator

This notebook uses **OGRePy** to construct the Einstein–Euler and general
first-order BDNK constitutive tensors for the spherically symmetric metric

$$
ds^2=-a(t,r)b(t,r)^2dt^2+2b(t,r)\,dt\,dr+r^2d\Omega^2.
$$

Conventions:

- $(x^0,x^1,x^2,x^3)=(t,r,\theta,\phi)$;
- signature $(-+++)$;
- $c=G=1$;
- $g_{\mu\nu}u^\mu u^\nu=-1$.

The ideal-fluid sector uses

$$
T^{\mu\nu}_{\mathrm{Euler}}=(\rho+p)u^\mu u^\nu+p g^{\mu\nu},
\qquad J^\mu=n u^\mu.
$$

The viscous sector implements the general first-order constitutive ansatz of
Bemfica–Disconzi–Noronha,

$$
J^\mu=\mathcal N u^\mu+\mathcal J^\mu,
$$

$$
T^{\mu\nu}_{\mathrm{BDNK}}=
\mathcal E u^\mu u^\nu+\mathcal P\Delta^{\mu\nu}
+u^\mu Q^\nu+u^\nu Q^\mu-2\eta\sigma^{\mu\nu}.
$$

**Scope warning.** The notebook verifies the tensor construction and its
algebraic projection identities. It does not claim that arbitrary transport
coefficients define a causal, stable, or strongly hyperbolic hydrodynamic
frame. Those properties require additional thermodynamic and frame-dependent
inequalities; see `VERIFICATION.md` and `references.bib`.


In [ ]:
# If needed, install first:
# !pip install OGRePy

import OGRePy as og
import sympy as sp
from sympy.core.function import AppliedUndef
from sympy.printing.latex import LatexPrinter

from IPython.display import display, Math
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "last_expr_or_assign"
og.options.friendly_errors = True

# Useful index symbols. We keep the symbolic index `mu` separate from the
# chemical potential variable `chem` below.
from OGRePy.abc import t, theta, phi, mu, nu, alpha, beta, lamda

# Coordinate r is nonnegative; this avoids abs(r) from sqrt(r^2)-type simplifications.
r = og.sym("r", nonnegative=True)
EF = og.Coordinates(t, r, theta, phi)
DIM = 4

# Full conservation-law and Einstein-BDNK component expansions are expensive.
# Leave this False for a quick reproducible run; set it True to export them.
BUILD_FULL_BDNK_EQUATIONS = False

## 0. Pretty LaTeX helpers

These helpers do not do any geometry. They only format the extracted SymPy components from OGRePy.

Important fix: component labels are printed as

$$
\left(T^{\mu\nu}_{\mathrm{Euler}}\right)_{00}
$$

instead of the invalid double-subscript form.

In [ ]:
LATEX_NAMES = {
    "theta": r"\theta",
    "phi": r"\phi",
    "rho": r"\rho",
    "chem": r"\mu",
    "mu_ch": r"\mu",
    "Temp": r"T",
    "U": r"U",
    "eps": r"\varepsilon",
    "P0": r"P",
    "n0": r"n",
    "eps_1": r"\varepsilon_1",
    "eps_2": r"\varepsilon_2",
    "eps_3": r"\varepsilon_3",
    "pi_1": r"\pi_1",
    "pi_2": r"\pi_2",
    "pi_3": r"\pi_3",
    "vartheta_1": r"\vartheta_1",
    "vartheta_2": r"\vartheta_2",
    "vartheta_3": r"\vartheta_3",
    "nu_1": r"\nu_1",
    "nu_2": r"\nu_2",
    "nu_3": r"\nu_3",
    "gamma_1": r"\gamma_1",
    "gamma_2": r"\gamma_2",
    "gamma_3": r"\gamma_3",
    "eta": r"\eta",
}

class CompactDerivativePrinter(LatexPrinter):
    """Compact LaTeX for long component formulas.

    - Undefined functions f(t,r) are printed as f.
    - Derivatives are printed as f_t, f_r, f_{tr}, etc.
    - Coefficient derivatives d eps_1(T,mu)/dT are printed compactly.
    """
    def _print_Symbol(self, expr):
        name = str(expr)
        return LATEX_NAMES.get(name, super()._print_Symbol(expr))

    def _undef_name(self, expr):
        name = expr.func.__name__
        return LATEX_NAMES.get(name, name)

    def _print_Function(self, expr, exp=None):
        if isinstance(expr, AppliedUndef):
            name = self._undef_name(expr)
            if exp is not None:
                return rf"{name}^{{{exp}}}"
            return name
        return super()._print_Function(expr, exp=exp)

    def _print_Derivative(self, expr):
        base = expr.expr
        if isinstance(base, AppliedUndef):
            base_latex = self._undef_name(base)
            indices = []
            for var, count in expr.variable_count:
                if isinstance(var, AppliedUndef):
                    var_latex = self._undef_name(var)
                else:
                    var_latex = self._print(var)
                indices.extend([var_latex] * count)
            return rf"{base_latex}_{{{''.join(indices)}}}"
        return super()._print_Derivative(expr)

def clatex(expr):
    return CompactDerivativePrinter().doprint(expr)

def emit_math(line):
    display(Math(line))

def component_label(label, *indices):
    suffix = ''.join(str(i) for i in indices)
    return rf"\left({label}\right)_{{{suffix}}}"

def as_expr(x):
    """Extract scalar component from a rank-0 OGRePy tensor, or return a SymPy expression."""
    if hasattr(x, "components"):
        return x.components(coords=EF, indices=(), warn=False)[0]
    return x

def show_scalar_equation(label, scalar_tensor_or_expr, rhs=0, simplify=False):
    expr = as_expr(scalar_tensor_or_expr)
    rhs = as_expr(rhs)
    if simplify:
        expr = og.s.simplify(expr)
        rhs = og.s.simplify(rhs)
    emit_math(rf"0 = {label} = {clatex(expr-rhs)}")

def show_vector_equations(label, tensor, indices=(1,), simplify=False, max_chars=None):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    for i in range(arr.shape[0]):
        expr = arr[i]
        if simplify:
            expr = og.s.simplify(expr)
        latex = clatex(expr)
        if max_chars is not None and len(latex) > max_chars:
            latex = latex[:max_chars] + r"\;\cdots"
        emit_math(rf"0 = {component_label(label, i)} = {latex}")

def show_nonzero_vector(label, tensor, indices=(1,), simplify=True, max_chars=None):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    for i in range(arr.shape[0]):
        expr = og.s.simplify(arr[i]) if simplify else arr[i]
        if expr != 0:
            latex = clatex(expr)
            if max_chars is not None and len(latex) > max_chars:
                latex = latex[:max_chars] + r"\;\cdots"
            emit_math(rf"{component_label(label, i)} = {latex}")

def show_nonzero_matrix(label, tensor, indices=(1, 1), simplify=True, max_chars=None):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    rows, cols = arr.shape
    for i in range(rows):
        for j in range(cols):
            expr = og.s.simplify(arr[i, j]) if simplify else arr[i, j]
            if expr != 0:
                latex = clatex(expr)
                if max_chars is not None and len(latex) > max_chars:
                    latex = latex[:max_chars] + r"\;\cdots"
                emit_math(rf"{component_label(label, i, j)} = {latex}")

def write_scalar_latex(filename, label, scalar_tensor_or_expr, simplify=False):
    expr = as_expr(scalar_tensor_or_expr)
    if simplify:
        expr = og.s.simplify(expr)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(rf"$$0 = {label} = {clatex(expr)}$$" + "\n")
    return filename

def write_vector_equations_latex(filename, label, tensor, indices=(1,), simplify=False):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(arr.shape[0]):
            expr = og.s.simplify(arr[i]) if simplify else arr[i]
            f.write(rf"$$0 = {component_label(label, i)} = {clatex(expr)}$$" + "\n\n")
    return filename


def write_vector_components_latex(filename, label, tensor, indices=(1,), simplify=False):
    """Write vector component identities, not equations set equal to zero."""
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(arr.shape[0]):
            expr = og.s.simplify(arr[i]) if simplify else arr[i]
            f.write(rf"$${component_label(label, i)} = {clatex(expr)}$$" + "\n\n")
    return filename

def write_matrix_latex(filename, label, tensor, indices=(1, 1), simplify=False, only_nonzero=True):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    rows, cols = arr.shape
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(rows):
            for j in range(cols):
                expr = og.s.simplify(arr[i, j]) if simplify else arr[i, j]
                if only_nonzero and expr == 0:
                    continue
                f.write(rf"$${component_label(label, i, j)} = {clatex(expr)}$$" + "\n\n")
    return filename

In [ ]:
def flatten_array(arr):
    if len(arr.shape) == 0:
        return [arr[()]]
    if len(arr.shape) == 1:
        return [arr[i] for i in range(arr.shape[0])]
    if len(arr.shape) == 2:
        return [arr[i, j] for i in range(arr.shape[0]) for j in range(arr.shape[1])]
    if len(arr.shape) == 3:
        return [arr[i, j, k] for i in range(arr.shape[0]) for j in range(arr.shape[1]) for k in range(arr.shape[2])]
    raise ValueError("flatten_array currently supports ranks 0,1,2,3 only.")

def all_zero_components(tensor, indices, simplify=True):
    arr = tensor.components(coords=EF, indices=indices, warn=False)
    vals = flatten_array(arr)
    if simplify:
        vals = [og.s.simplify(v) for v in vals]
    return all(v == 0 for v in vals), vals

## 1. Geometry: metric and velocity

We use the same Eddington–Finkelstein-type ansatz:

$$
g_{\mu\nu}=\begin{pmatrix}
-a b^2 & b & 0 & 0\\
b&0&0&0\\
0&0&r^2&0\\
0&0&0&r^2\sin^2\theta
\end{pmatrix}.
$$

The radial four-velocity is normalized by construction:

$$
u^\mu=\left(U,\frac{ab^2U^2-1}{2bU},0,0\right).
$$

In [ ]:
# Metric functions.
a = og.func("a")(t, r)
b = og.func("b")(t, r)
U = og.func("U")(t, r)

Metric = og.Metric(
    coords=EF,
    components=og.s.Matrix([
        [-a*b**2, b, 0, 0],
        [b,       0, 0, 0],
        [0,       0, r**2, 0],
        [0,       0, 0, r**2 * og.s.sin(theta)**2],
    ]),
    symbol="g",
)

Velocity = og.Tensor(
    metric=Metric,
    coords=EF,
    indices=(1,),
    components=og.s.Array([
        U,
        (a*b**2*U**2 - 1) / (2*b*U),
        0,
        0,
    ]),
    symbol="u",
)

Metric.show()
Velocity.show()

In [ ]:
# Normalization check: u_mu u^mu = -1.
NormU = og.calc(
    formula=Velocity(mu) @ Velocity(mu),
    symbol=r"u_\mu u^\mu",
)

emit_math(rf"u_\mu u^\mu = {clatex(og.s.simplify(NormU.components(coords=EF, indices=(), warn=False)[0]))}")

## 2. Testing Euler equations

We construct

$$
T^{\mu\nu}_{\mathrm{Euler}}=(\rho+p)u^\mu u^\nu+p g^{\mu\nu},
\qquad p=k^2\rho,
$$

and

$$
J^\mu_{\mathrm{Euler}}=n u^\mu.
$$

The Euler equations are

$$
\nabla_\mu T^{\mu\nu}_{\mathrm{Euler}}=0,
\qquad
\nabla_\mu J^\mu_{\mathrm{Euler}}=0.
$$

I also include the coupled Einstein equation

$$
G_{\mu\nu}+\Lambda g_{\mu\nu}=8\pi T_{\mu\nu}.
$$

In [ ]:
rho = og.func("rho")(t, r)
n_baryon = og.func("n")(t, r)
k = og.sym("k")
Lam = og.sym("Lambda")

p_euler = k**2 * rho

EulerStress = og.calc(
    formula=(rho + p_euler) * (Velocity(mu) @ Velocity(nu)) + p_euler * Metric(mu, nu),
    symbol=r"T_{\mathrm{Euler}}",
)
EulerStress.default_indices = (1, 1)

EulerCurrent = og.calc(
    formula=n_baryon * Velocity(mu),
    symbol=r"J_{\mathrm{Euler}}",
)
EulerCurrent.default_indices = (1,)

EulerEnergyEq = og.calc(
    formula=og.CovariantD(mu) @ EulerStress(mu, nu),
    symbol=r"\nabla T_{\mathrm{Euler}}",
)
EulerNumberEq = og.calc(
    formula=og.CovariantD(mu) @ EulerCurrent(mu),
    symbol=r"\nabla J_{\mathrm{Euler}}",
)

EinsteinEulerEq = og.calc(
    formula=Metric.einstein(mu, nu) + Lam * Metric(mu, nu) - 8 * og.s.pi * EulerStress(mu, nu),
    symbol=r"\mathcal{G}_{\mathrm{Euler}}",
)
EinsteinEulerEq.default_indices = (-1, -1)

In [ ]:
print("Nonzero components of T_Euler^{mu nu}:")
show_nonzero_matrix(r"T_{\mathrm{Euler}}^{\mu\nu}", EulerStress, indices=(1, 1), simplify=True)

print("Euler equations: 0 = nabla_mu T_Euler^{mu nu}")
show_vector_equations(r"\nabla_\mu T_{\mathrm{Euler}}^{\mu\nu}", EulerEnergyEq, indices=(1,), simplify=False)

print("Euler number equation: 0 = nabla_mu J_Euler^mu")
show_scalar_equation(r"\nabla_\mu J_{\mathrm{Euler}}^\mu", EulerNumberEq, simplify=False)

print("Einstein-Euler equations: 0 = G_{mu nu} + Lambda g_{mu nu} - 8 pi T_{mu nu}")
show_nonzero_matrix(r"\mathcal{G}_{\mathrm{Euler},\mu\nu}", EinsteinEulerEq, indices=(-1, -1), simplify=False, max_chars=5000)

In [ ]:
write_matrix_latex("ogrepy_euler_stress_components.tex", r"T_{\mathrm{Euler}}^{\mu\nu}", EulerStress, indices=(1, 1), simplify=False)
write_vector_equations_latex("ogrepy_euler_energy_equations.tex", r"\nabla_\mu T_{\mathrm{Euler}}^{\mu\nu}", EulerEnergyEq, indices=(1,), simplify=False)
write_scalar_latex("ogrepy_euler_number_equation.tex", r"\nabla_\mu J_{\mathrm{Euler}}^\mu", EulerNumberEq, simplify=False)
write_matrix_latex("ogrepy_einstein_euler_equations.tex", r"\mathcal{G}_{\mathrm{Euler},\mu\nu}", EinsteinEulerEq, indices=(-1, -1), simplify=False)

print("Wrote Euler output .tex files.")

## 3. BDNK construction

Dependencies:

$$
T=T(t,r),\qquad \mu=\mu(t,r),\qquad X=\frac{\mu}{T}.
$$

Equation-of-state functions:

$$
\varepsilon=\varepsilon(T,\mu),\qquad P=P(T,\mu),\qquad n=n(T,\mu).
$$

Transport coefficients:

$$
\varepsilon_i,\ \pi_i,\ \vartheta_i,\ \nu_i,\ \gamma_i,\ \eta
\quad\text{depend only on}\quad (T,\mu).
$$

The BDNK scalar corrections are implemented as

$$
\mathcal E=\varepsilon
+\varepsilon_1\frac{DT}{T}
+\varepsilon_2\Theta
+\varepsilon_3D\left(\frac{\mu}{T}\right),
$$

$$
\mathcal P=P
+\pi_1\frac{DT}{T}
+\pi_2\Theta
+\pi_3D\left(\frac{\mu}{T}\right),
$$

$$
\mathcal N=n
+\nu_1\frac{DT}{T}
+\nu_2\Theta
+\nu_3D\left(\frac{\mu}{T}\right),
$$

where

$$
D=u^\alpha\nabla_\alpha,
\qquad
\Theta=\nabla_\alpha u^\alpha.
$$

The BDNK dissipative vectors are implemented as

$$
Q^\mu=\vartheta_1\frac{\Delta^{\mu\alpha}\nabla_\alpha T}{T}
+\vartheta_2 a^\mu
+\vartheta_3\Delta^{\mu\alpha}\nabla_\alpha\left(\frac{\mu}{T}\right),
$$

$$
\mathcal J^\mu=\gamma_1\frac{\Delta^{\mu\alpha}\nabla_\alpha T}{T}
+\gamma_2 a^\mu
+\gamma_3\Delta^{\mu\alpha}\nabla_\alpha\left(\frac{\mu}{T}\right),
$$

with the **correct acceleration**

$$
a^\mu=u^\alpha\nabla_\alpha u^\mu.
$$

This is the main correction relative to the placeholder version: do **not** replace $a^\mu$ by $\Theta u^\mu$.

The coefficients below are kept symbolic. Thermodynamic consistency of the
general ansatz imposes, in the notation used here,
$\vartheta_1=\vartheta_2$ and $\gamma_1=\gamma_2$. Causality, stability, and
strong hyperbolicity require further restrictions and a specified equation of
state; the calculator does not impose those restrictions automatically.


In [ ]:
# Thermodynamic variables with explicit dependence.
Temp = og.func("Temp")(t, r)
chem = og.func("mu_ch")(t, r)
X_expr = chem / Temp

TempScalar = og.Tensor(metric=Metric, coords=EF, indices=(), components=[Temp], symbol="T")
XScalar = og.Tensor(metric=Metric, coords=EF, indices=(), components=[X_expr], symbol=r"\mu/T")

# Equation of state functions.
eps0 = og.func("eps")(Temp, chem)
P0 = og.func("P0")(Temp, chem)
n0 = og.func("n0")(Temp, chem)

# BDNK transport coefficients.
eps1 = og.func("eps_1")(Temp, chem)
eps2 = og.func("eps_2")(Temp, chem)
eps3 = og.func("eps_3")(Temp, chem)

pi1 = og.func("pi_1")(Temp, chem)
pi2 = og.func("pi_2")(Temp, chem)
pi3 = og.func("pi_3")(Temp, chem)

vartheta1 = og.func("vartheta_1")(Temp, chem)
vartheta2 = og.func("vartheta_2")(Temp, chem)
vartheta3 = og.func("vartheta_3")(Temp, chem)

nu1 = og.func("nu_1")(Temp, chem)
nu2 = og.func("nu_2")(Temp, chem)
nu3 = og.func("nu_3")(Temp, chem)

gamma1 = og.func("gamma_1")(Temp, chem)
gamma2 = og.func("gamma_2")(Temp, chem)
gamma3 = og.func("gamma_3")(Temp, chem)

eta = og.func("eta")(Temp, chem)

### 3.1 OGRePy geometric building blocks

The following are now built with OGRePy tensor operations:

$$
\Delta^{\mu\nu}=g^{\mu\nu}+u^\mu u^\nu,
$$

$$
\Theta=\nabla_\alpha u^\alpha,
$$

$$
DT=u^\alpha\nabla_\alpha T,
\qquad
DX=u^\alpha\nabla_\alpha X,
$$

$$
a^\mu=u^\alpha\nabla_\alpha u^\mu,
$$

$$
\nabla_\perp^\mu T=\Delta^{\mu\alpha}\nabla_\alpha T,
\qquad
\nabla_\perp^\mu X=\Delta^{\mu\alpha}\nabla_\alpha X.
$$

For the shear, I use the equivalent upper-index form

$$
\sigma^{\mu\nu}
=\frac12\left(
\Delta^{\mu\alpha}\nabla_\alpha u^\nu
+\Delta^{\nu\alpha}\nabla_\alpha u^\mu
\right)
-\frac13\Delta^{\mu\nu}\Theta.
$$

In [ ]:
Delta = og.calc(
    formula=Metric(mu, nu) + (Velocity(mu) @ Velocity(nu)),
    symbol=r"\Delta",
)
Delta.default_indices = (1, 1)

Theta = og.calc(
    formula=og.CovariantD(alpha) @ Velocity(alpha),
    symbol=r"\Theta",
)
Theta_expr = og.s.simplify(Theta.components(coords=EF, indices=(), warn=False)[0])

DTemp = og.calc(
    formula=Velocity(alpha) @ (og.CovariantD(alpha) @ TempScalar()),
    symbol=r"DT",
)
DTemp_expr = og.s.simplify(DTemp.components(coords=EF, indices=(), warn=False)[0])

DX = og.calc(
    formula=Velocity(alpha) @ (og.CovariantD(alpha) @ XScalar()),
    symbol=r"DX",
)
DX_expr = og.s.simplify(DX.components(coords=EF, indices=(), warn=False)[0])

GradTempPerp = og.calc(
    formula=Delta(mu, alpha) @ (og.CovariantD(alpha) @ TempScalar()),
    symbol=r"\nabla_\perp T",
)
GradTempPerp.default_indices = (1,)

GradXPerp = og.calc(
    formula=Delta(mu, alpha) @ (og.CovariantD(alpha) @ XScalar()),
    symbol=r"\nabla_\perp X",
)
GradXPerp.default_indices = (1,)

Acceleration = og.calc(
    formula=Velocity(alpha) @ (og.CovariantD(alpha) @ Velocity(mu)),
    symbol="a",
)
Acceleration.default_indices = (1,)

Shear = og.calc(
    formula=(
        og.s.Rational(1, 2) * (
            Delta(mu, alpha) @ (og.CovariantD(alpha) @ Velocity(nu))
            + Delta(nu, alpha) @ (og.CovariantD(alpha) @ Velocity(mu))
        )
        - og.s.Rational(1, 3) * Theta_expr * Delta(mu, nu)
    ),
    symbol=r"\sigma",
)
Shear.default_indices = (1, 1)

In [ ]:
print("Compact BDNK geometric scalars:")
emit_math(rf"\Theta = {clatex(Theta_expr)}")
emit_math(rf"DT = {clatex(DTemp_expr)}")
emit_math(rf"DX = {clatex(DX_expr)}")

print("Nonzero components of a^mu:")
show_nonzero_vector(r"a^\mu", Acceleration, indices=(1,), simplify=False, max_chars=4000)

print("Nonzero components of sigma^{mu nu}:")
show_nonzero_matrix(r"\sigma^{\mu\nu}", Shear, indices=(1, 1), simplify=False, max_chars=4000)

### 3.2 BDNK corrected scalars, vectors, current, and stress tensor

Now define

$$
\mathcal E,\quad \mathcal P,\quad \mathcal N,
\quad Q^\mu,
\quad \mathcal J^\mu,
\quad T^{\mu\nu}_{\mathrm{BDNK}},
\quad J^\mu_{\mathrm{BDNK}}.
$$

In [ ]:
Ecal_expr = eps0 + eps1 * DTemp_expr / Temp + eps2 * Theta_expr + eps3 * DX_expr
Pcal_expr = P0   + pi1  * DTemp_expr / Temp + pi2  * Theta_expr + pi3  * DX_expr
Ncal_expr = n0   + nu1  * DTemp_expr / Temp + nu2  * Theta_expr + nu3  * DX_expr

Q = og.calc(
    formula=(vartheta1 / Temp) * GradTempPerp(mu)
            + vartheta2 * Acceleration(mu)
            + vartheta3 * GradXPerp(mu),
    symbol="Q",
)
Q.default_indices = (1,)

Jdiss = og.calc(
    formula=(gamma1 / Temp) * GradTempPerp(mu)
            + gamma2 * Acceleration(mu)
            + gamma3 * GradXPerp(mu),
    symbol=r"\mathcal{J}",
)
Jdiss.default_indices = (1,)

BDNKStress = og.calc(
    formula=Ecal_expr * (Velocity(mu) @ Velocity(nu))
            + Pcal_expr * Delta(mu, nu)
            + (Velocity(mu) @ Q(nu))
            + (Velocity(nu) @ Q(mu))
            - 2 * eta * Shear(mu, nu),
    symbol=r"T_{\mathrm{BDNK}}",
)
BDNKStress.default_indices = (1, 1)

BDNKCurrent = og.calc(
    formula=Ncal_expr * Velocity(mu) + Jdiss(mu),
    symbol=r"J_{\mathrm{BDNK}}",
)
BDNKCurrent.default_indices = (1,)

In [ ]:
print("BDNK scalar corrections:")
emit_math(rf"\mathcal{{E}} = {clatex(Ecal_expr)}")
emit_math(rf"\mathcal{{P}} = {clatex(Pcal_expr)}")
emit_math(rf"\mathcal{{N}} = {clatex(Ncal_expr)}")

print("Nonzero components of Q^mu:")
show_nonzero_vector(r"Q^\mu", Q, indices=(1,), simplify=False, max_chars=5000)

print("Nonzero components of J_diss^mu:")
show_nonzero_vector(r"\mathcal{J}^\mu", Jdiss, indices=(1,), simplify=False, max_chars=5000)

### 3.3 BDNK equations

The BDNK PDEs are

$$
\nabla_\mu T^{\mu\nu}_{\mathrm{BDNK}}=0,
\qquad
\nabla_\mu J^\mu_{\mathrm{BDNK}}=0.
$$

The coupled Einstein–BDNK equations are

$$
G_{\mu\nu}+\Lambda g_{\mu\nu}=8\pi T^{\mathrm{BDNK}}_{\mu\nu}.
$$

The full component formulas can be extremely long, so the notebook writes them to `.tex` files. You can also display them by setting `PRINT_FULL_BDNK = True`.

In [ ]:
if BUILD_FULL_BDNK_EQUATIONS:
    BDNKEnergyEq = og.calc(
        formula=og.CovariantD(mu) @ BDNKStress(mu, nu),
        symbol=r"\nabla T_{\mathrm{BDNK}}",
    )

    BDNKNumberEq = og.calc(
        formula=og.CovariantD(mu) @ BDNKCurrent(mu),
        symbol=r"\nabla J_{\mathrm{BDNK}}",
    )

    EinsteinBDNKEq = og.calc(
        formula=Metric.einstein(mu, nu) + Lam * Metric(mu, nu) - 8 * og.s.pi * BDNKStress(mu, nu),
        symbol=r"\mathcal{G}_{\mathrm{BDNK}}",
    )
    EinsteinBDNKEq.default_indices = (-1, -1)

    print("Built the full BDNK conservation and Einstein equations.")
else:
    BDNKEnergyEq = None
    BDNKNumberEq = None
    EinsteinBDNKEq = None
    print("Skipped full BDNK PDE expansions. Set BUILD_FULL_BDNK_EQUATIONS = True to build them.")


## 4. Correctness checks

These checks verify geometric consistency of the BDNK building blocks:

$$
\Delta^{\mu\nu}u_\nu=0,
\qquad
u_\mu a^\mu=0,
$$

$$
\sigma^{\mu\nu}=\sigma^{\nu\mu},
\qquad
g_{\mu\nu}\sigma^{\mu\nu}=0,
\qquad
u_\mu\sigma^{\mu\nu}=0.
$$

If the formulas were constructed consistently, these should simplify to zero / `True`.

In [ ]:
def scalar_zero(tensor, correction=0):
    value = tensor.components(coords=EF, indices=(), warn=False)[0]
    return og.s.simplify(value + correction) == 0


checks = {}

VelocityNorm = og.calc(formula=Velocity(mu) @ Velocity(mu), symbol=r"u\cdot u")
checks[r"u_mu u^mu = -1"] = scalar_zero(VelocityNorm, correction=1)

DeltaUCheck = og.calc(formula=Delta(mu, nu) @ Velocity(nu), symbol=r"\Delta u")
checks[r"Delta^{mu nu} u_nu = 0"] = all_zero_components(DeltaUCheck, indices=(1,))[0]

AccelOrthogonality = og.calc(formula=Velocity(mu) @ Acceleration(mu), symbol=r"u\cdot a")
checks[r"u_mu a^mu = 0"] = scalar_zero(AccelOrthogonality)

ShearSymmetryCheck = og.calc(
    formula=Shear(mu, nu) - Shear(nu, mu), symbol=r"\sigma-\sigma^T"
)
checks[r"sigma^{mu nu} = sigma^{nu mu}"] = all_zero_components(
    ShearSymmetryCheck, indices=(1, 1)
)[0]

ShearTraceCheck = og.calc(formula=Metric(mu, nu) @ Shear(mu, nu), symbol=r"tr\,\sigma")
checks[r"g_{mu nu} sigma^{mu nu} = 0"] = scalar_zero(ShearTraceCheck)

ShearOrthogonality = og.calc(formula=Velocity(mu) @ Shear(mu, nu), symbol=r"u\sigma")
checks[r"u_mu sigma^{mu nu} = 0"] = all_zero_components(
    ShearOrthogonality, indices=(1,)
)[0]

QOrthogonality = og.calc(formula=Velocity(mu) @ Q(mu), symbol=r"u\cdot Q")
checks[r"u_mu Q^mu = 0"] = scalar_zero(QOrthogonality)

JdissOrthogonality = og.calc(formula=Velocity(mu) @ Jdiss(mu), symbol=r"u\cdot J_diss")
checks[r"u_mu Jdiss^mu = 0"] = scalar_zero(JdissOrthogonality)

NumberProjection = og.calc(formula=Velocity(mu) @ BDNKCurrent(mu), symbol=r"u\cdot J_BDNK")
checks[r"-u_mu J_BDNK^mu = Ncal"] = scalar_zero(NumberProjection, correction=Ncal_expr)

CurrentSpatialProjection = og.calc(
    formula=(Delta(mu, alpha) @ BDNKCurrent(alpha)) - Jdiss(mu),
    symbol=r"\Delta J-J_diss",
)
checks[r"Delta^mu_nu J_BDNK^nu = Jdiss^mu"] = all_zero_components(
    CurrentSpatialProjection, indices=(1,)
)[0]

StressSymmetry = og.calc(
    formula=BDNKStress(mu, nu) - BDNKStress(nu, mu), symbol=r"T-T^T"
)
checks[r"T_BDNK^{mu nu} = T_BDNK^{nu mu}"] = all_zero_components(
    StressSymmetry, indices=(1, 1)
)[0]

EnergyProjection = og.calc(
    formula=Velocity(mu) @ BDNKStress(mu, nu) @ Velocity(nu), symbol=r"uTu"
)
checks[r"u_mu u_nu T_BDNK^{mu nu} = Ecal"] = scalar_zero(
    EnergyProjection, correction=-Ecal_expr
)

PressureProjection = og.calc(
    formula=Delta(mu, nu) @ BDNKStress(mu, nu), symbol=r"\Delta T"
)
checks[r"(1/3) Delta_mn T_BDNK^{mn} = Pcal"] = scalar_zero(
    PressureProjection, correction=-3 * Pcal_expr
)

HeatFluxProjection = og.calc(
    formula=(Delta(mu, alpha) @ Velocity(beta) @ BDNKStress(alpha, beta)) + Q(mu),
    symbol=r"\Delta uT+Q",
)
checks[r"-Delta^mu_a u_b T_BDNK^{ab} = Q^mu"] = all_zero_components(
    HeatFluxProjection, indices=(1,)
)[0]

for description, passed in checks.items():
    print(f"{description}: {passed}")

failed = [description for description, passed in checks.items() if not passed]
if failed:
    raise AssertionError(f"Failed constitutive checks: {failed}")


## 5. Export full formulas

This writes all untruncated formulas to Markdown/KaTeX-friendly `.tex` files using `$$...$$` display math.

In [ ]:
# Euler output.
write_matrix_latex("ogrepy_euler_stress_components.tex", r"T_{\mathrm{Euler}}^{\mu\nu}", EulerStress, indices=(1, 1), simplify=False)
write_vector_equations_latex("ogrepy_euler_energy_equations.tex", r"\nabla_\mu T_{\mathrm{Euler}}^{\mu\nu}", EulerEnergyEq, indices=(1,), simplify=False)
write_scalar_latex("ogrepy_euler_number_equation.tex", r"\nabla_\mu J_{\mathrm{Euler}}^\mu", EulerNumberEq, simplify=False)
write_matrix_latex("ogrepy_einstein_euler_equations.tex", r"\mathcal{G}_{\mathrm{Euler},\mu\nu}", EinsteinEulerEq, indices=(-1, -1), simplify=False)

# BDNK constitutive components.
write_vector_components_latex("ogrepy_bdnk_Q_components.tex", r"Q^\mu", Q, indices=(1,), simplify=False)
write_vector_components_latex("ogrepy_bdnk_Jdiss_components.tex", r"\mathcal{J}^\mu", Jdiss, indices=(1,), simplify=False)
write_matrix_latex("ogrepy_bdnk_stress_components.tex", r"T_{\mathrm{BDNK}}^{\mu\nu}", BDNKStress, indices=(1, 1), simplify=False)

if BUILD_FULL_BDNK_EQUATIONS:
    write_vector_equations_latex("ogrepy_bdnk_energy_equations.tex", r"\nabla_\mu T_{\mathrm{BDNK}}^{\mu\nu}", BDNKEnergyEq, indices=(1,), simplify=False)
    write_scalar_latex("ogrepy_bdnk_number_equation.tex", r"\nabla_\mu J_{\mathrm{BDNK}}^\mu", BDNKNumberEq, simplify=False)
    write_matrix_latex("ogrepy_einstein_bdnk_equations.tex", r"\mathcal{G}_{\mathrm{BDNK},\mu\nu}", EinsteinBDNKEq, indices=(-1, -1), simplify=False)

print("Wrote available OGRePy Euler and BDNK formulas to .tex files.")
